In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["HUGGINGFACEHUB_API_TOKEN"]=os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [2]:
from langchain.chat_models import init_chat_model

model=init_chat_model("groq:llama-3.3-70b-versatile")


Summarization middleware

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware,HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage,HumanMessage

#msg based summarization
agent1=create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [8]:
#create thread
config={
    "configurable":{"thread_id":"test-1"}
}

In [9]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent1.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='3c73f566-6f30-4e9c-93e5-0c434fc1eea7'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.011813449, 'completion_tokens_details': None, 'prompt_time': 0.001951073, 'prompt_tokens_details': None, 'queue_time': 0.052658232, 'total_time': 0.013764522}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9017-c96f-73a2-ba4e-dd23ffd6ecb6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='3c73f566-6f30-4e9c-93e5-0c434fc1eea7'), AIMessa

In [16]:
from langchain_core.tools import tool
@tool
def get_hotel(city:str)->str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent2=create_agent(
    model=model,
    tools=[get_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",600),
            keep=("tokens",200)
        )
    ]
)

In [19]:
config={"configurable":{"thread_id":"test-1"}}

In [20]:
# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [21]:
cities = ["Paris", "London", "Tokyo"]

for city in cities:
    response = agent2.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~137 tokens, 6 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='e66bdbc8-5d8e-46a9-adcc-8d5aa51849ad'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ytwmttv0x', 'function': {'arguments': '{"city":"Paris"}', 'name': 'get_hotel'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 226, 'total_tokens': 241, 'completion_time': 0.040168033, 'completion_tokens_details': None, 'prompt_time': 0.014004816, 'prompt_tokens_details': None, 'queue_time': 0.163179969, 'total_time': 0.054172849}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f901f-312c-7512-aceb-6c91cb7217f4-0', tool_calls=[{'name': 'get_hotel', 'args': {'city': 'Paris'}, 'id': 'ytwmttv0x', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inp

Human in the loop middleware

In [23]:
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [24]:
agent3=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [25]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent3.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [26]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='58542a80-fb99-4ebd-bbc0-183cf0504810'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ytmfx46j0', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.063580327, 'completion_tokens_details': None, 'prompt_time': 0.187760953, 'prompt_tokens_details': None, 'queue_time': 0.610837167, 'total_time': 0.25134128}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f902b-2b1d-7be0-989b-71554c819696-0', tool_calls=[{'name': 'send_email_tool', 'args': {'bo

In [27]:
from langgraph.types import Command

# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent3.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Please note that the function call was executed successfully. 

If you'd like to read the email, you would need to know the email ID. If you have the email ID, you can use the read_email_tool function.


In [28]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='58542a80-fb99-4ebd-bbc0-183cf0504810'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ytmfx46j0', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.063580327, 'completion_tokens_details': None, 'prompt_time': 0.187760953, 'prompt_tokens_details': None, 'queue_time': 0.610837167, 'total_time': 0.25134128}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f902b-2b1d-7be0-989b-71554c819696-0', tool_calls=[{'name': 'send_email_tool', 'args': {'bo